In [1]:
import os
import pandas as pd
import re
import ast
import pandas as pd
from getpass import getpass
from google import genai


df = pd.read_csv('sales_data.csv')
print('Shape:', df.shape)
display(df.head())

Shape: (1000, 14)


,Product_ID,Sale_Date,Sales_Rep,Region,Sales_Amount,Quantity_Sold,Product_Category,Unit_Cost,Unit_Price,Customer_Type,Discount,Payment_Method,Sales_Channel,Region_and_Sales_Rep
0,1052,2023-02-03,Bob,North,5053.97,18,Furniture,152.75,267.22,Returning,0.09,Cash,Online,North-Bob
1,1093,2023-04-21,Bob,West,4384.02,17,Furniture,3816.39,4209.44,Returning,0.11,Cash,Retail,West-Bob
2,1015,2023-09-21,David,South,4631.23,30,Food,261.56,371.40,Returning,0.20,Bank Transfer,Retail,South-David
3,1072,2023-08-24,Bob,South,2167.94,39,Clothing,4330.03,4467.75,New,0.02,Credit Card,Retail,South-Bob
4,1061,2023-03-24,Charlie,East,3750.20,13,Electronics,637.37,692.71,New,0.08,Credit Card,Online,East-Charlie


In [3]:
import os
from getpass import getpass

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass(
        "Enter your Gemini API key: "
    )

MODEL = "gemini-3.8-flash"

In [4]:
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

ALLOWED_WORDS = [
    'df', 'head', 'tail', 'shape', 'columns', 'dtypes',
    'describe', 'mean', 'sum', 'min', 'max', 'count',
    'value_counts', 'groupby', 'sort_values', 'reset_index',
    'to_string', 'round', 'iloc', 'loc', 'query', 'unique',
    'nunique', 'len'
]

BLOCKED = [
    'import ', 'open(', 'exec(', 'eval(', '__', 'os.',
    'sys.', 'subprocess', 'read_csv', 'to_csv',
    'requests', 'lambda'
]


def ask_gemini(prompt, max_tokens=300):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"max_output_tokens": max_tokens}
    )
    if not response.text:
        raise ValueError("Gemini returned an empty response.")
    return response.text.strip()


def make_code(question):
    prompt = f"""You answer questions about a pandas DataFrame called df.

Columns and types:
{df.dtypes.to_string()}

User question: {question}

Return ONLY one safe Python expression using df, such as:
- df["units"].sum()
- df.groupby("product")["units"].sum()
- df.describe()

Do not use imports, files, internet, assignments, or arbitrary Python."""

    text = ask_gemini(prompt)
    text = re.sub(
        r'^```(?:python)?\s*|\s*```$', '', text
    ).strip()
    return text


def safe_run(expression):
    # Validate the expression's syntax and allowed operations.
    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError:
        raise ValueError("Gemini returned invalid Python syntax.")

    allowed_names = {"df", "len"}
    allowed_methods = set(ALLOWED_WORDS) - {"df", "len"}

    for node in ast.walk(tree):
        if isinstance(node, ast.Name):
            if node.id not in allowed_names:
                raise ValueError("Disallowed name in expression.")
        elif isinstance(node, ast.Call):
            if not isinstance(node.func, ast.Attribute):
                if not (
                    isinstance(node.func, ast.Name)
                    and node.func.id == "len"
                ):
                    raise ValueError("Disallowed function call.")
            elif node.func.attr not in allowed_methods:
                raise ValueError("Disallowed method.")
        elif isinstance(node, ast.Attribute):
            if node.attr.startswith("_") or node.attr not in allowed_methods:
                raise ValueError("Disallowed attribute.")
        elif isinstance(node, ast.Subscript):
            pass
        elif isinstance(node, (ast.Expression, ast.Load, ast.Constant,
                               ast.BinOp, ast.UnaryOp, ast.Compare,
                               ast.BoolOp, ast.And, ast.Or, ast.Not,
                               ast.Add, ast.Sub, ast.Mult, ast.Div,
                               ast.Mod, ast.USub, ast.UAdd, ast.Eq,
                               ast.NotEq, ast.Lt, ast.LtE, ast.Gt,
                               ast.GtE, ast.In, ast.NotIn, ast.List,
                               ast.Tuple, ast.Slice)):
            pass
        else:
            raise ValueError(
                f"Disallowed Python syntax: {type(node).__name__}"
            )

    return eval(
        compile(tree, "<safe_expression>", "eval"),
        {"__builtins__": {"len": len}},
        {"df": df}
    )


def answer_csv(question):
    expression = make_code(question)

    print("Pandas code generated by Gemini:")
    print(expression)

    result = safe_run(expression)

    print("\nResult:")
    if isinstance(result, (pd.DataFrame, pd.Series)):
        display(result)
    else:
        print(result)

    explanation_prompt = f"""Explain this CSV analysis simply in 2 sentences.

Question: {question}
Pandas expression: {expression}
Result: {result}

Do not invent facts beyond the result."""

    explanation = ask_gemini(explanation_prompt, max_tokens=250)
    print("\nExplanation:", explanation)



## 5. Ask questions

In [ ]:
question = input('Ask a question about the CSV: ')
answer_csv(question)